# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a self-contained, step-by-step workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [10.71728/senscience.y7m0-f273](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset and print its high-level metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")
print(f"\nIdentifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Spatial Coverage: {metadata.spatialCoverage}")
print(f"Temporal Coverage: {metadata.temporalCoverage}")


## 2. Data Overview
Review the available record sets, their `@id`s, fields, and columns. You may need to dig into the Croissant schema for specific structure—here we enumerate what's discoverable from the loaded dataset.

In [ ]:
# List all record sets (by @id) and corresponding fields in the dataset
record_sets = list(dataset.record_sets())

if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rs in record_sets:
        print(f"\nRecord Set Name: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
        print("  Columns:")
        for column in getattr(rs, 'columns', []):
            print(f"    - {column.name} (@id: {getattr(column, 'id', 'N/A')})")

# Also print a summary number of record_sets found (by @id):
print("\nAvailable record sets:")
record_set_ids = [rs.id for rs in record_sets]
print(record_set_ids)


## 3. Data Extraction
Load records for each record set into a DataFrame using the `@id`. You can reference fields by their `@id` as well.

If there are no record sets, skip extraction; if there are, demonstrate on the first one.

In [ ]:
# Prepare DataFrames for each record set
dataframes = {}
if not record_sets:
    print("No record sets available for extraction.")
else:
    for rs in record_sets:
        records = list(dataset.records(record_set=rs.id))  # reference by @id
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Loaded {len(df)} records from record set: {rs.name} (@id: {rs.id})")
    # Example: print column names and first 5 records from the first record set
    main_record_set_id = record_sets[0].id
    print("\nColumns in first record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    print("\nPreview of first record set:")
    display(dataframes[main_record_set_id].head())
    

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping. All references use field `@id`s.

The following uses an example numeric field—ensure to update the field `@id` to match one from your chosen record set.

In [ ]:
# If there is at least one record set and the DataFrame is not empty, try EDA
import numpy as np

if record_sets:
    # Choose main record set and select numeric field and grouping field by `@id`
    main_rs = record_sets[0]
    df = dataframes[main_rs.id]
    
    # Heuristically select a numeric field (from fields list)
    numeric_field_id = None
    for field in main_rs.fields:
        if field.data_type in ('Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number'):
            numeric_field_id = field.id
            break
    if numeric_field_id is not None and numeric_field_id in df.columns:
        # Set a threshold using quantiles if possible, else fallback
        # We'll use the 10th percentile as example
        threshold = df[numeric_field_id].dropna().quantile(0.1) if not df[numeric_field_id].dropna().empty else 10

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by the first non-numeric field (categorical)
        group_field_id = None
        for field in main_rs.fields:
            if field.id != numeric_field_id and df[field.id].dtype == np.object_:
                group_field_id = field.id
                break
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped (mean) {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping EDA.")
    else:
        print("No numeric field found for EDA in the main record set.")
else:
    print("No record sets available for EDA.")


## 5. Visualization
Visualize the numeric field's distribution and, if applicable, its relationship to the grouping variable.

In [ ]:
# Basic matplotlib/seaborn visualization if fields available
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field_id and numeric_field_id in df.columns:
    # Histogram of numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    # If group_field_id is set, make a boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion

- We loaded the FAIR^2 dataset using its Croissant schema URL and explored its metadata.
- We listed available record sets and their field and column `@id`s.
- A record set was loaded into a DataFrame, enabling filtering, normalization, and group-wise analysis on a selected numeric field.
- We visualized field distributions and their relation to key grouping fields.

**Next steps:** Refer to the schema's full documentation or the dataset metadata for deeper, field- or domain-specific exploration. The examples can be extended to fit a full analysis pipeline, leveraging the robust field `@id` referencing to ensure reproducibility.
